# YOLO Pollinator Detector

Trains YOLO11 to detect and classify pollinators directly in full Arctic field images.

**Classes:** `bumblebee` · `fly` · `butterfly` · `other`

## Data structure expected
```
dataset/
├── data.yaml          ← generated by this notebook
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/         ← YOLO format .txt files from CVAT
    └── val/
```

CVAT export: **YOLO 1.1** format. Class order in CVAT must match `CLASSES` below.

## Strategy for imbalanced data
We use:
- `copy_paste` augmentation — copies rare-class objects into other images
- `mixup` augmentation — blends images to expose model to rare classes more
- Two-stage training: freeze backbone first, then unfreeze all layers
- External data (optional): mix in web bumblebee, butterfly and other images 


In [ ]:
# Install ultralytics if needed
# !pip install ultralytics --quiet

import shutil
import json
import random
from pathlib import Path

import numpy as np
import yaml
from ultralytics import YOLO

print('Ultralytics version:', __import__('ultralytics').__version__)


## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Root folder containing images/ and labels/ from CVAT export
DATASET_ROOT = Path('dataset')
# Optional: extra annotated images from web / Roboflow for rare classes
# Set to None to skip. Folder structure same as DATASET_ROOT.
EXTRA_DATA_ROOT = None   # e.g. Path('roboflow_bees')
MODEL_OUT_DIR = Path('runs/pollinator')

# ── Classes ────────────────────────────────────────────────────────────────
# Must match the class order used in CVAT annotation
CLASSES = ['bumblebee', 'fly', 'butterfly', 'other']

# ── Model ──────────────────────────────────────────────────────────────────
# yolo11n = smallest/fastest, good for limited data
# yolo11s = slightly larger, better if GPU allows
# yolo11m = medium, use if you have 1000+ bbox per class
MODEL_SIZE = 'yolo26n.pt'

# ── Training settings ──────────────────────────────────────────────────────
IMG_SIZE   = 640
BATCH      = 16
SEED       = 42
VAL_FRAC   = 0.2

# Stage 1: frozen backbone (head only) — avoids destroying pretrained features
EPOCHS_STAGE1 = 30
LR_STAGE1     = 1e-3
FREEZE_LAYERS = 10   # freeze first N backbone layers

# Stage 2: full fine-tune with small learning rate — helps adapt backbone to our data
EPOCHS_STAGE2 = 70
LR_STAGE2     = 1e-4

# ── Augmentation ───────────────────────────────────────────────────────────
# copy_paste: copies rare-class bbox into other images — helps minority classes a lot
# mixup: blends two images — regularises, helps rare classes appear more
COPY_PASTE = 0.3
MIXUP      = 0.1

# ── Verify data ────────────────────────────────────────────────────────────
def count_bboxes(labels_dir):
    counts = {i: 0 for i in range(len(CLASSES))}
    for txt in Path(labels_dir).glob('*.txt'):
        for line in txt.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                cls = int(parts[0])
                if cls < len(CLASSES):
                    counts[cls] += 1
    return counts

for split in ('train', 'val'):
    ldir = DATASET_ROOT / 'labels' / split
    idir = DATASET_ROOT / 'images' / split
    if not ldir.exists():
        print(f'WARNING: {ldir} not found')
        continue
    n_imgs = len(list(idir.glob('*.jpg'))) + len(list(idir.glob('*.png')))
    counts = count_bboxes(ldir)
    print(f'{split}: {n_imgs} images')
    for i, cls in enumerate(CLASSES):
        flag = '  ← low' if counts[i] < 100 else ''
        print(f'  {cls:15}: {counts[i]:>5} bbox{flag}')


## Prepare Dataset

Splits images into train/val if not already split, merges extra data, and generates `data.yaml`.

In [ ]:
def auto_split(dataset_root, val_frac, seed):
    """If train/val split doesn't exist, create it from all images in images/."""
    img_dir = dataset_root / 'images'
    lbl_dir = dataset_root / 'labels'
    if (img_dir / 'train').exists():
        print('train/val split already exists, skipping auto-split')
        return
    print('No train/val split found — auto-splitting...')
    all_imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    random.seed(seed)
    random.shuffle(all_imgs)
    n_val = max(1, int(len(all_imgs) * val_frac))
    splits = {'val': all_imgs[:n_val], 'train': all_imgs[n_val:]}
    for split, imgs in splits.items():
        (img_dir / split).mkdir(parents=True, exist_ok=True)
        (lbl_dir / split).mkdir(parents=True, exist_ok=True)
        for img in imgs:
            shutil.copy(img, img_dir / split / img.name)
            lbl = lbl_dir / (img.stem + '.txt')
            if lbl.exists():
                shutil.copy(lbl, lbl_dir / split / lbl.name)
    print(f'Split done: train={len(splits["train"])} val={len(splits["val"])}')


def merge_extra_data(extra_root, dataset_root):
    """Copy extra annotated images into train split."""
    if extra_root is None or not extra_root.exists():
        return
    print(f'Merging extra data from {extra_root}...')
    n = 0
    for split in ('train', ''):
        idir = extra_root / 'images' / split if split else extra_root / 'images'
        ldir = extra_root / 'labels' / split if split else extra_root / 'labels'
        if not idir.exists():
            continue
        for img in list(idir.glob('*.jpg')) + list(idir.glob('*.png')):
            dst_img = dataset_root / 'images' / 'train' / f'extra_{img.name}'
            dst_lbl = dataset_root / 'labels' / 'train' / f'extra_{img.stem}.txt'
            shutil.copy(img, dst_img)
            lbl = ldir / (img.stem + '.txt')
            if lbl.exists():
                shutil.copy(lbl, dst_lbl)
            n += 1
    print(f'Merged {n} extra images into train')


def write_yaml(dataset_root, classes):
    yaml_path = dataset_root / 'data.yaml'
    cfg = {
        'path': str(dataset_root.resolve()),
        'train': 'images/train',
        'val':   'images/val',
        'nc':    len(classes),
        'names': classes,
    }
    yaml_path.write_text(yaml.dump(cfg, default_flow_style=False))
    print(f'data.yaml written to {yaml_path}')
    return yaml_path


auto_split(DATASET_ROOT, VAL_FRAC, SEED)
merge_extra_data(EXTRA_DATA_ROOT, DATASET_ROOT)
YAML_PATH = write_yaml(DATASET_ROOT, CLASSES)
print(f'\nYAML preview:')
print(YAML_PATH.read_text())


## Stage 1 — Frozen Backbone Training

Train only the detection head. Backbone stays frozen to preserve pretrained features.
Good when data is limited — prevents backbone from forgetting general visual features.

In [ ]:
model = YOLO(MODEL_SIZE)
print(f'Model: {MODEL_SIZE}')

results_s1 = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS_STAGE1,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR_STAGE1,
    freeze=FREEZE_LAYERS,
    patience=15,
    copy_paste=COPY_PASTE,
    mixup=MIXUP,
    seed=SEED,
    project=str(MODEL_OUT_DIR),
    name='stage1_frozen',
    exist_ok=True,
    verbose=True,
)

stage1_best = MODEL_OUT_DIR / 'stage1_frozen' / 'weights' / 'best.pt'
print(f'\nStage 1 done. Best weights: {stage1_best}')
print(f'Stage 1 mAP50: {results_s1.results_dict.get("metrics/mAP50(B)", "n/a"):.3f}')


## Stage 2 — Full Fine-Tune

Unfreeze all layers and continue training with a small learning rate.
Adapts the backbone to Arctic field image characteristics.

In [ ]:
# Load best Stage 1 checkpoint
model_s2 = YOLO(str(stage1_best))

results_s2 = model_s2.train(
    data=str(YAML_PATH),
    epochs=EPOCHS_STAGE2,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR_STAGE2,
    freeze=0,           # unfreeze all layers
    patience=20,
    copy_paste=COPY_PASTE,
    mixup=MIXUP,
    seed=SEED,
    project=str(MODEL_OUT_DIR),
    name='stage2_finetune',
    exist_ok=True,
    verbose=True,
)

stage2_best = MODEL_OUT_DIR / 'stage2_finetune' / 'weights' / 'best.pt'
print(f'\nStage 2 done. Best weights: {stage2_best}')
print(f'Stage 2 mAP50: {results_s2.results_dict.get("metrics/mAP50(B)", "n/a"):.3f}')


## Evaluation

In [ ]:
model_eval = YOLO(str(stage2_best))
metrics = model_eval.val(data=str(YAML_PATH), imgsz=IMG_SIZE, batch=BATCH)

print('\n--- Per-class results ---')
box = metrics.box
for i, cls in enumerate(CLASSES):
    try:
        p  = box.p[i]
        r  = box.r[i]
        f1 = 2 * p * r / max(1e-8, p + r)
        ap = box.ap50[i]
        print(f'{cls:15}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP50={ap:.3f}')
    except (IndexError, AttributeError):
        print(f'{cls:15}  (no detections)')

print(f'\nmAP50:    {box.map50:.3f}')
print(f'mAP50-95: {box.map:.3f}')

# Save results summary
summary = {
    'model': str(stage2_best),
    'classes': CLASSES,
    'mAP50': float(box.map50),
    'mAP50_95': float(box.map),
    'per_class': {}
}
for i, cls in enumerate(CLASSES):
    try:
        p = float(box.p[i]); r = float(box.r[i])
        summary['per_class'][cls] = {
            'precision': p, 'recall': r,
            'f1': 2*p*r/max(1e-8, p+r),
            'ap50': float(box.ap50[i])
        }
    except (IndexError, AttributeError):
        summary['per_class'][cls] = None

(MODEL_OUT_DIR / 'results.json').write_text(json.dumps(summary, indent=2))
print(f'Saved to {MODEL_OUT_DIR}/results.json')


## Inference — Run on a Single Image or Folder

In [ ]:
import cv2
import matplotlib.pyplot as plt

CONF_THRESHOLD = 0.25   # minimum confidence to show a detection

# ── Single image ──────────────────────────────────────────────────────────
def predict_image(model, img_path, conf=CONF_THRESHOLD):
    results = model.predict(str(img_path), conf=conf, verbose=False)
    r = results[0]
    print(f'\n{Path(img_path).name}: {len(r.boxes)} detections')
    for box in r.boxes:
        cls_name = CLASSES[int(box.cls)]
        conf_val = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print(f'  {cls_name:15} conf={conf_val:.2f}  bbox=[{x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f}]')
    # Show annotated image
    img_annotated = r.plot()
    img_rgb = cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(Path(img_path).name)
    plt.tight_layout()
    plt.show()
    return r


# ── Batch inference on a folder — writes results to CSV ───────────────────
def predict_folder(model, img_folder, conf=CONF_THRESHOLD, out_csv=None):
    import csv
    imgs = list(Path(img_folder).glob('*.jpg')) + list(Path(img_folder).glob('*.png'))
    rows = []
    for img_path in sorted(imgs):
        results = model.predict(str(img_path), conf=conf, verbose=False)
        r = results[0]
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            rows.append({
                'image': img_path.name,
                'class': CLASSES[int(box.cls)],
                'confidence': round(float(box.conf), 4),
                'x1': round(x1), 'y1': round(y1),
                'x2': round(x2), 'y2': round(y2),
            })
    if out_csv:
        with open(out_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['image','class','confidence','x1','y1','x2','y2'])
            writer.writeheader(); writer.writerows(rows)
        print(f'Results saved to {out_csv}')
    print(f'\nTotal detections: {len(rows)} across {len(imgs)} images')
    for cls in CLASSES:
        n = sum(1 for row in rows if row['class'] == cls)
        print(f'  {cls:15}: {n}')
    return rows


# ── Example usage ─────────────────────────────────────────────────────────
model_final = YOLO(str(stage2_best))

# Test on one val image
val_imgs = list((DATASET_ROOT / 'images' / 'val').glob('*.jpg'))
if val_imgs:
    predict_image(model_final, val_imgs[0])
else:
    print('No val images found')
